In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from matplotlib_venn import venn2, venn2_circles, venn2_unweighted
from matplotlib_venn import venn3, venn3_circles
import re
%matplotlib inline
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

<a id='sessions'></a>
# Tables/Sessions

* [Load and prepare tables](#load)
* [Pseudogene Detection and Annotation](#pseudogene)
* [Data analysis: General Profile](#data)

## Anvio metabolism MAG genomic fasta
<a id='load'></a>
### Load and prepare tables
* [Sessions list](#sessions)

In [3]:
modules = pd.read_csv('kegg-metabolism_modules.txt', sep="\t")
modules.rename(columns={"stepwise_module_completeness": "step_compl", "pathwise_module_completeness": "path_compl",
                        "enzymes_unique_to_module": "enzymes_uniq","proportion_unique_enzymes_present": "prop_enzymes_uniq"}, inplace=True)
modules = modules.drop(columns=['module_class', 'stepwise_module_is_complete','pathwise_module_is_complete',
                                'per_step_copy_numbers','gene_caller_ids_in_module','pathwise_copy_number','stepwise_copy_number',
                                'genome_name', 'unique_enzymes_hit_counts']).drop_duplicates().sort_values(by=['module_category', 'module_subcategory']).reset_index(drop=True)
modules.head(2)

,module,module_name,module_category,module_subcategory,module_definition,step_compl,path_compl,prop_enzymes_uniq,enzymes_uniq,enzyme_hits_in_module,warnings
0,M00022,"Shikimate pathway, phosphoenolpyruvate + erythrose-4P => chorismate",Amino acid metabolism,Aromatic amino acid metabolism,"(K01626,K03856,K13853) (((K01735,K13829) ((K03785,K03786) K00014,K13832)),K13830) ((K00891,K13829) (K00800,K24018),K13830) K01736",0.750000,0.80,1.0,"K00800,K00891,K01626,K01736,K03786","K00800,K00891,K01626,K01626,K01736,K03786",NaN
1,M00023,"Tryptophan biosynthesis, chorismate => tryptophan",Amino acid metabolism,Aromatic amino acid metabolism,"(((K01657+K01658,K13503,K13501,K01656) K00766),K13497) (((K01817,K24017) (K01656,K01609)),K13498,K13501) (K01695+(K01696,K06001),K01694)",0.666667,0.75,1.0,"K00766,K01695,K01696,K13498","K00766,K01695,K01696,K13498",NaN


In [4]:
modules_complete = modules[((modules['step_compl'] == 1.0) & (modules['path_compl'] == 1.0))].drop_duplicates().sort_values(by=['module_category','module_subcategory']).reset_index(drop=True)
modules_complete.drop(columns=['module_definition','warnings','step_compl','path_compl'], inplace=True)
#modules_complete.to_csv('Kegg_metabolism_modules_complete.tsv', sep='\t', index=False)
modules_complete_list = modules_complete['module'].unique().tolist()
modules_incomplete = modules[(((modules['step_compl'] >= 0.75) & (modules['step_compl'] < 1)) | ((modules['path_compl'] >= 0.75) & (modules['path_compl'] < 1)))].drop_duplicates().sort_values(by=['module_category', 'module_subcategory']).reset_index(drop=True)
modules_incomplete.drop(columns=['module_definition','warnings'], inplace=True)
#modules_incomplete.to_csv('Kegg_metabolism_modules_incomplete.tsv', sep='\t', index=False)
modules_incomplete_list = modules_incomplete['module'].unique().tolist()
modules_low = modules[((modules['step_compl'] < 0.75) & (modules['path_compl'] < 0.75))].drop_duplicates().sort_values(by=['module_category', 'module_subcategory']).reset_index(drop=True)
modules_low.drop(columns=['module_definition','warnings'], inplace=True)
#modules_low.to_csv('Kegg_metabolism_modules_low.tsv', sep='\t', index=False)
modules_low_list = modules_low['module'].unique().tolist()
modules_incomplete.head(2)

,module,module_name,module_category,module_subcategory,step_compl,path_compl,prop_enzymes_uniq,enzymes_uniq,enzyme_hits_in_module
0,M00022,"Shikimate pathway, phosphoenolpyruvate + erythrose-4P => chorismate",Amino acid metabolism,Aromatic amino acid metabolism,0.750000,0.80,1.0,"K00800,K00891,K01626,K01736,K03786","K00800,K00891,K01626,K01626,K01736,K03786"
1,M00023,"Tryptophan biosynthesis, chorismate => tryptophan",Amino acid metabolism,Aromatic amino acid metabolism,0.666667,0.75,1.0,"K00766,K01695,K01696,K13498","K00766,K01695,K01696,K13498"


In [92]:
hits = pd.read_csv('Sodalis_hits.txt', sep="\t")
hits.rename(columns={"modules_with_enzyme": "module"}, inplace=True)
hits = hits.drop(['genome_name','gene_caller_id'], axis=1).drop_duplicates().sort_values(by=['enzyme']).reset_index(drop=True)
#hits.to_csv('Kegg_hits_unique_enzymes.tsv', sep='\t', index=False)

## Emapper MAG Prokka-annotation, amino acids
### Get only annotations with KO significantly assigned

In [ ]:
def load_and_edit_emapper(file_path):
    """
    Load and preprocess an emapper annotation file.
    Args:
        file_path (str): Path to the emapper annotation file.
    Returns:
        pd.DataFrame: Processed DataFrame with relevant columns and transformations.
    """
    import pandas as pd

    # Load the emapper annotation file
    emapper_df = pd.read_csv(file_path, sep="\t", skiprows=4)
    # Rename columns for consistency
    emapper_df.rename(columns={"#query": "query"}, inplace=True)
    # Filter rows based on KEGG_ko and evalue
    emapper_df = emapper_df[(emapper_df['KEGG_ko'] != '-') & (emapper_df['evalue'] <= 1e-6)]
    # Split and explode KEGG_Module and KEGG_ko columns
    emapper_df['KEGG_Module'] = emapper_df['KEGG_Module'].str.split(',')
    emapper_df = emapper_df.explode('KEGG_Module')
    emapper_df['KEGG_ko'] = emapper_df['KEGG_ko'].str.split(',')
    emapper_df = emapper_df.explode('KEGG_ko')
    # Remove 'ko:' prefix from KEGG_ko values
    emapper_df['KEGG_ko'] = emapper_df['KEGG_ko'].str.replace('ko:', '', regex=False)
    # Rename KEGG_Module column to module
    emapper_df.rename(columns={"KEGG_Module": "module"}, inplace=True)
    # Create a new column 'taxa_scope' by splitting the 'eggNOG_OGs' column
    emapper_df['taxa_scope'] = emapper_df['eggNOG_OGs'].apply(
        lambda x: x.split('|')[-1] if isinstance(x, str) else None
    )
    # Select relevant columns
    emapper_df = emapper_df[
        ['query', 'COG_category', 'Preferred_name', 'EC', 'KEGG_ko', 'module',
         'CAZy', 'PFAMs', 'Description', 'taxa_scope', 'KEGG_Pathway', 'BRITE']
    ].drop_duplicates().reset_index(drop=True)

    return emapper_df

In [40]:
Sodalis_emapper = load_and_edit_emapper("/home/marlaux/Tati/pseudogenes_metabolic_integration2/input/prokka_out/emapper/mRsS25-Chr1.emapper.annotations")
Sodalis_emapper.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
0,MRSS_00001,G,pgm,5.4.2.2,K01835,M00549,-,"PGM_PMM_I,PGM_PMM_II,PGM_PMM_III,PGM_PMM_IV",Phosphoglucomutase,Gammaproteobacteria,"ko00010,ko00030,ko00052,ko00230,ko00500,ko00520,ko00521,ko01100,ko01110,ko01120,ko01130,map00010,map00030,map00052,map00230,map00500,map00520,map00521,map01100,map01110,map01120,map01130","ko00000,ko00001,ko00002,ko01000"
1,MRSS_00002,F,guaB,1.1.1.205,K00088,M00050,-,"CBS,IMPDH,NMO","Catalyzes the conversion of inosine 5'-phosphate (IMP) to xanthosine 5'-phosphate (XMP), the first committed and rate- limiting step in the de novo synthesis of guanine nucleotides, and therefore plays an important role in the regulation of cell growth",Gammaproteobacteria,"ko00230,ko00983,ko01100,ko01110,map00230,map00983,map01100,map01110","ko00000,ko00001,ko00002,ko01000,ko04147"


In [8]:
# Create KEGG KO/query to KEGG mapping file
Sodalis_kos_to_kegg_mapper = Sodalis_emapper[['query','KEGG_ko']].drop_duplicates()
Sodalis_kos_to_kegg_mapper.to_csv('Sodalis_kos_to_kegg_mapper.tsv', sep='\t', index=False, header=False)

In [10]:
Sodalis_emapper_ko_list = Sodalis_emapper['KEGG_ko'].unique().tolist()
Sodalis_emapper_modules_list = Sodalis_emapper['module'].unique().tolist()
# Write the list to a text file, one ID per line
with open('Sodalis_kos_list.txt', 'w') as f:
    for item in Sodalis_emapper_ko_list:
        f.write(f"{item}\n")
with open('Sodalis_modules_list.txt', 'w') as f:
    for item in Sodalis_emapper_modules_list:
        f.write(f"{item}\n")

### Get KEGG KO and modules names
- Run fetch_ko_definitions.py and fetch_module_definitions.py scripts<p>
- This will create the Sodalis_kos_description.txt (ko_symbol and ko_name) and Sodalis_modules_description.txt ('module_name','module_class','module_pathway')

In [ ]:
!python3 tools/fetch_ko_definitions.py --ko-list Sodalis_kos_list.txt
!python3 tools/fetch_module_definitions.py --module-list Sodalis_modules_list.txt

List of IDs has been exported to Sodalis_kos_list.txt


In [41]:
Sodalis_kos_df = pd.read_csv('Sodalis_kos_definitions.csv', sep=",")
Sodalis_modules_df = pd.read_csv('Sodalis_modules_definitions.csv', sep=",")
Sodalis_emapper = pd.merge(Sodalis_emapper,Sodalis_kos_df[['KEGG_ko','ko_symbol','ko_name']], on=['KEGG_ko'], how='left')
Sodalis_emapper = pd.merge(Sodalis_emapper,Sodalis_modules_df[['module','module_name','module_class','module_pathway']], on='module', how='left')
Sodalis_emapper = Sodalis_emapper[['query','COG_category','Preferred_name','EC','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','Description','CAZy','PFAMs','KEGG_Pathway','BRITE','taxa_scope']]
Sodalis_emapper.loc[(Sodalis_emapper['module'] != '-') & (Sodalis_emapper['module_name'].isna()), 'module'] = 'No_entry_found'
Sodalis_emapper.loc[(Sodalis_emapper['module'] == 'No_entry_found'), ['module_name','module_class','module_pathway']] = '-'
Sodalis_emapper.head(2)
#Sodalis_emapper.to_csv("Sodalis_emapper_annotations_with_ko_module_names.tsv", sep="\t", index=False)

,query,COG_category,Preferred_name,EC,KEGG_ko,ko_symbol,ko_name,module,module_name,module_class,module_pathway,Description,CAZy,PFAMs,KEGG_Pathway,BRITE,taxa_scope
0,MRSS_00001,G,pgm,5.4.2.2,K01835,pgm,phosphoglucomutase [EC:5.4.2.2],M00549,"UDP-Glc biosynthesis, Glc => UDP-Glc",Pathway modules; Glycan metabolism; Nucleotide sugar biosynthesis,map00520 Amino sugar and nucleotide sugar metabolism,Phosphoglucomutase,-,"PGM_PMM_I,PGM_PMM_II,PGM_PMM_III,PGM_PMM_IV","ko00010,ko00030,ko00052,ko00230,ko00500,ko00520,ko00521,ko01100,ko01110,ko01120,ko01130,map00010,map00030,map00052,map00230,map00500,map00520,map00521,map01100,map01110,map01120,map01130","ko00000,ko00001,ko00002,ko01000",Gammaproteobacteria
1,MRSS_00002,F,guaB,1.1.1.205,K00088,"IMPDH, guaB",IMP dehydrogenase [EC:1.1.1.205],M00050,"Guanine ribonucleotide biosynthesis, IMP => GDP,GTP",Pathway modules; Nucleotide metabolism; Purine metabolism,map00230 Purine metabolism,"Catalyzes the conversion of inosine 5'-phosphate (IMP) to xanthosine 5'-phosphate (XMP), the first committed and rate- limiting step in the de novo synthesis of guanine nucleotides, and therefore plays an important role in the regulation of cell growth",-,"CBS,IMPDH,NMO","ko00230,ko00983,ko01100,ko01110,map00230,map00983,map01100,map01110","ko00000,ko00001,ko00002,ko01000,ko04147",Gammaproteobacteria


## Use a close symbiont reference to check the metabolic completeness
### Emapper Reference Sodalis glossinidius morsitans

In [42]:
Sodalis_ref_emapper = load_and_edit_emapper("/home/marlaux/Tati/pseudogenes_metabolic_integration2/input/Sodalis_reference/Sodalis_reference.emapper.annotations")
Sodalis_ref_emapper.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
0,WP_000429386.1,C,atpE,-,K02110,M00157,-,ATP-synt_C,"F(1)F(0) ATP synthase produces ATP from ADP in the presence of a proton or sodium gradient. F-type ATPases consist of two structural domains, F(1) containing the extramembraneous catalytic core and F(0) containing the membrane proton channel, linked together by a central stalk and a peripheral stalk. During catalysis, ATP synthesis in the catalytic domain of F(1) is coupled via a rotary mechanism of the central stalk subunits to proton translocation",Gammaproteobacteria,"ko00190,ko00195,ko01100,map00190,map00195,map01100","ko00000,ko00001,ko00002,ko00194"
1,WP_000664222.1,L,-,2.1.1.72,K06223,-,-,MethyltransfD12,Site-specific DNA-methyltransferase (Adenine-specific),Gammaproteobacteria,"ko03430,map03430","ko00000,ko00001,ko01000,ko02048,ko03032,ko03400"


In [14]:
#Sodalis_ref_kos_list = Sodalis_ref_emapper['KEGG_ko'].unique().tolist()
#Sodalis_ref_modules_list = Sodalis_ref_emapper['module'].unique().tolist()
# Write the list to a text file, one ID per line
#with open('Sodalis_ref_kos_list.txt', 'w') as f:
#    for item in Sodalis_ref_kos_list:
#        f.write(f"{item}\n")
# Run tools/fetch_ko_definitions.py and tools/fetch_module_definitions.py to get KO and modules definitions

In [43]:
#sodalis_ref_kos_df = pd.read_csv('Sodalis_ref_kos_definitions.csv', sep=",")
#Sodalis_ref_modules_df = pd.read_csv('Sodalis_ref_module_definitions.csv', sep=",")
Sodalis_ref_emapper = pd.merge(Sodalis_ref_emapper,Sodalis_kos_df[['KEGG_ko','ko_symbol','ko_name']], on=['KEGG_ko'], how='left')
Sodalis_ref_emapper = pd.merge(Sodalis_ref_emapper,Sodalis_modules_df[['module','module_name','module_class','module_pathway']], on='module', how='left')
Sodalis_ref_emapper = Sodalis_ref_emapper[['query','COG_category','Description','Preferred_name','EC','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','KEGG_Pathway','BRITE','CAZy','PFAMs','taxa_scope']]
Sodalis_ref_emapper.loc[(Sodalis_ref_emapper['module'] != '-') & (Sodalis_ref_emapper['module_name'].isna()), 'module'] = 'No_entry_found'
Sodalis_ref_emapper.loc[(Sodalis_ref_emapper['module'] == 'No_entry_found'), ['module_name','module_class','module_pathway']] = '-'
# Create KEGG KO/query to KEGG mapping file
#Sodalis_ref_kos_to_kegg_mapper = Sodalis_ref_emapper[['query','KEGG_ko']].drop_duplicates()
#Sodalis_ref_kos_to_kegg_mapper.to_csv('Sodalis_ref_kos_to_kegg_mapper.tsv', sep='\t', index=False, header=False)
#Sodalis_ref_emapper.to_csv("Sodalis_ref_emapper_annotations_with_ko_module_names.tsv", sep="\t", index=False)

## Emapper annotation Eukaryota-Paraneoptera
### Use the unbinned contigs to annotate eukaryota hits

In [32]:
Eukaryota_annotations = load_and_edit_emapper("/home/marlaux/Tati/pseudogenes_metabolic_integration2/metabolism/emapper_eukaryota/Eukaryota.emapper.annotations")
Eukaryota_annotations.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
0,k141_1000_0,A,SF3B3,-,K12830,M00352,-,"CPSF_A,MMS1_N","splicing factor 3b, subunit 3",Cetartiodactyla,"ko03040,map03040","ko00000,ko00001,ko00002,ko03021,ko03036,ko03041"
1,k141_100027_0,I,-,1.2.1.84,K13356,-,-,"NAD_binding_4,Sterile",Catalyzes the reduction of fatty acyl-CoA to fatty alcohols,Hymenoptera,"ko00073,ko04146,ko04212,map00073,map04146,map04212","ko00000,ko00001,ko01000"


In [33]:
# Eukaryota_annotations['taxa_scope'].unique() # 
insecta = ['Hymenoptera', 'Paraneoptera', 'Drosophilidae', 'Insecta', 'Nematocera', 'Lepidoptera', 'Diptera']
insecta_annotations = Eukaryota_annotations[Eukaryota_annotations['taxa_scope'].isin(insecta)]
insecta_annotations.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
1,k141_100027_0,I,-,1.2.1.84,K13356,-,-,"NAD_binding_4,Sterile",Catalyzes the reduction of fatty acyl-CoA to fatty alcohols,Hymenoptera,"ko00073,ko04146,ko04212,map00073,map04146,map04212","ko00000,ko00001,ko01000"
2,k141_100050_0,K,exd,-,K09355,-,-,"Homeobox,PBC",PBC domain,Paraneoptera,"ko04927,ko04934,ko05202,map04927,map04934,map05202","ko00000,ko00001,ko03000"


In [12]:
#insecta_annotations_kos_list = insecta_annotations['KEGG_ko'].unique().tolist()
#insecta_annotations_modules_list = insecta_annotations['module'].unique().tolist()
# Write the list to a text file, one ID per line
#with open('insecta_annotations_kos_list.txt', 'w') as f:
#    for item in insecta_annotations_kos_list:
#        f.write(f"{item}\n")
# Run tools/fetch_ko_definitions.py and tools/fetch_module_definitions.py to get KO and modules definitions

## Use a host reference to check the metabolic completeness
### Emapper Reference Eukaryota Cyamophila willieti

In [34]:
Cyamophila_annotations = load_and_edit_emapper("/home/marlaux/Tati/pseudogenes_metabolic_integration2/metabolism/emapper_Cyamophila/Cyamophila_willieti.emapper.annotations")
#Cyamophila_annotations[['query','KEGG_ko']].drop_duplicates().to_csv('Cyamophila_KEGG_Mapper.tsv', sep="\t", index=False, header=False)
Cyamophila_annotations.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
0,KAL1445863.1,S,-,-,K09228,-,-,"zf-AD,zf-C2H2","Zinc finger, C2H2 type",Paraneoptera,-,"ko00000,ko03000"
1,KAL1445864.1,S,-,-,K09228,-,-,"zf-AD,zf-C2H2",Zinc-finger associated domain (zf-AD),Hymenoptera,-,"ko00000,ko03000"


<a id='pseudogene'></a>
## Pseudogene Detection and Annotation
* [Sessions list](#sessions)

### Pseudogene Detection and Annotation Methodology

To detect and annotate pseudogenes, we applied a systematic approach combining computational tools and customized scripts:


**Customized Scripts**:
   - `detect_stops_compare_length.py`: Identified CDS with internal stop codons and compared the alignment lengths of predicted proteins to reference to detect truncations
   - `remap_coverage_variants.sh`: Run remap to get coverage and variants information
   - `compare_pseudogenes_with_reference.sh`: Compare the pseudogene candidates with the original reference genome using blastn (genomic sequences)
   - `incorporate_annotations.py`: Combine alignment results, read mapping data and incorporate functional annotations
   - `compare_manual_with_pseudofinder.py`: Run Pseudofinder tool and compare results 

### Pseudogenes: manual detection

In [150]:
def load_and_edit_tables_with_kegg(file_path):
    """
    Load and preprocess a table with multi-valued KEGG columns.
    Args:
        file_path (str): Path to table..
    Returns:
        pd.DataFrame: Processed DataFrame with relevant columns and transformations.
    """
    import pandas as pd

    # Load the file
    df = pd.read_csv(file_path, sep="\t")
    # Filter out rows with missing KEGG_ko values
    df = df[df['KEGG_ko'] != '-']
    # Split and explode KEGG_Module and KEGG_ko columns
    df['KEGG_Module'] = df['KEGG_Module'].str.split(',')
    df = df.explode('KEGG_Module')
    df['KEGG_ko'] = df['KEGG_ko'].str.split(',')
    df = df.explode('KEGG_ko')
    # Remove 'ko:' prefix from KEGG_ko values
    df['KEGG_ko'] = df['KEGG_ko'].str.replace('ko:', '', regex=False)
    # Rename KEGG_Module column to module
    df.rename(columns={"KEGG_Module": "module"}, inplace=True)

    return df

In [151]:
manual_pseudogenes = load_and_edit_tables_with_kegg("/home/marlaux/Tati/pseudogenes_metabolic_integration2/output/final_pseudogene_candidates_disruptions_with_annotations.tsv")
manual_pseudogenes = pd.merge(manual_pseudogenes,
                                          Sodalis_emapper[['query','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','KEGG_Pathway','BRITE']],
                                          on=['query','KEGG_ko','module'],
                                          how='left').drop_duplicates()
manual_pseudogenes.head(2)

,query,Disruption,Coverage,Cov_comp,Variant_Type,DP,Preferred_name,COG_category,EC,KEGG_ko,module,Description,ko_symbol,ko_name,module_name,module_class,module_pathway,KEGG_Pathway,BRITE
0,MRSS_00006,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00019,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00290 Valine, leucine and isoleucine biosynthesis","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"
1,MRSS_00006,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00036,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Leucine degradation, leucine => acetoacetate + acetyl-CoA",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00280 Valine, leucine and isoleucine degradation","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"


In [199]:
comparison_manual_pseudofinder = load_and_edit_tables_with_kegg("/home/marlaux/Tati/pseudogenes_metabolic_integration2/output/comparison_manual_pseudofinder_report.tsv")
comparison_manual_pseudofinder = pd.merge(comparison_manual_pseudofinder,
                                          Sodalis_emapper[['query','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','KEGG_Pathway','BRITE']],
                                          on=['query','KEGG_ko','module'],
                                          how='left').drop_duplicates()
#comparison_manual_pseudofinder.head(2)

### Count the number of different reasons for the pseudogene candidate detection and filter candidate with minimum 2 reasons

In [200]:
# Create a new column called reasons_count
comparison_manual_pseudofinder_reasons = comparison_manual_pseudofinder[['query', 'Pseudofinder_reason', 'Disruption', 'KEGG_ko']].drop_duplicates()

# Group by 'query' and 'KEGG_ko' to count reasons and disruptions
comparison_manual_pseudofinder_reasons = (
    comparison_manual_pseudofinder_reasons
    .groupby(['query', 'KEGG_ko'], as_index=False)
    .agg(
        pseudo_reasons_count=('Pseudofinder_reason', lambda x: len(set(x))),  # Count unique Pseudofinder_reason
        Pseudofinder_reasons=('Pseudofinder_reason', lambda x: ';'.join(set(x))),  # Join unique Pseudofinder_reason
        disruptions_count=('Disruption', lambda x: sum(len(set(d.split(';'))) for d in x if isinstance(d, str)))  # Count unique disruptions
    )
)

# Calculate total reasons count
comparison_manual_pseudofinder_reasons['reasons_count'] = (
    comparison_manual_pseudofinder_reasons['pseudo_reasons_count'] +
    comparison_manual_pseudofinder_reasons['disruptions_count']
)

# Merge to pseudofinder_only_emapper DataFrame
comparison_manual_pseudofinder = pd.merge(
    comparison_manual_pseudofinder,
    comparison_manual_pseudofinder_reasons[['query', 'KEGG_ko', 'reasons_count', 'Pseudofinder_reasons']],
    on=['query', 'KEGG_ko'],
    how='left'
)

# Reorder columns
comparison_manual_pseudofinder.drop(columns=['Pseudofinder_reason'], inplace=True)
comparison_manual_pseudofinder.insert(2, 'reasons_count', comparison_manual_pseudofinder.pop('reasons_count'))
comparison_manual_pseudofinder.insert(3, 'Pseudofinder_reasons', comparison_manual_pseudofinder.pop('Pseudofinder_reasons'))

# filter minimum 2 reasons
comparison_manual_pseudo_filter = comparison_manual_pseudofinder[comparison_manual_pseudofinder['reasons_count'] >= 2]
# Display the result
comparison_manual_pseudo_filter.head(2)

,query,Identification_Source,reasons_count,Pseudofinder_reasons,Disruption,Coverage,Cov_comp,Variant_Type,DP,Preferred_name,COG_category,EC,KEGG_ko,module,Description,ko_symbol,ko_name,module_name,module_class,module_pathway,KEGG_Pathway,BRITE
0,MRSS_00006,Match,6,Intergenic region with 15 blast hits.;ORF is 23.4%25 of the average length of hits to this gene.,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00019,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00290 Valine, leucine and isoleucine biosynthesis","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"
1,MRSS_00006,Match,6,Intergenic region with 15 blast hits.;ORF is 23.4%25 of the average length of hits to this gene.,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00036,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Leucine degradation, leucine => acetoacetate + acetyl-CoA",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00280 Valine, leucine and isoleucine degradation","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"


In [201]:
pseudofinder_only = load_and_edit_tables_with_kegg("/home/marlaux/Tati/pseudogenes_metabolic_integration2/output/comparison_pseudofinder_only_report.tsv")
pseudofinder_only.drop(columns=['Identification_Source'], inplace=True)
pseudofinder_only = pd.merge(pseudofinder_only,
                             Sodalis_emapper[['query','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','KEGG_Pathway','BRITE']],
                             on=['query','KEGG_ko','module'],
                             how='left').drop_duplicates()
#pseudofinder_only.head(2)

In [ ]:
# create a new column called reasons_count
# Group by 'query' and 'KEGG_ko', count reasons, and join them with ';'
pseudofinder_only_reasons = (
    pseudofinder_only
    .groupby(['query', 'KEGG_ko'], as_index=False)
    .agg(
        reasons_count=('Pseudofinder_reason', lambda x: len(set(x))),  # Count unique Pseudofinder_reason
        Pseudofinder_reasons=('Pseudofinder_reason', lambda x: ';'.join(set(x))),  # Join unique Pseudofinder_reason
    )
)
# merge to pseudofinder_only DataFrame
pseudofinder_only.drop(columns=['Pseudofinder_reason'], inplace=True)
pseudofinder_only = pd.merge(pseudofinder_only,pseudofinder_only_reasons[['query','KEGG_ko','reasons_count','Pseudofinder_reasons']], on=['query','KEGG_ko'], how='left')

# Reorder columns
pseudofinder_only.insert(1, 'reasons_count', pseudofinder_only.pop('reasons_count'))
pseudofinder_only.insert(2, 'Pseudofinder_reasons', pseudofinder_only.pop('Pseudofinder_reasons'))

# filter minimum 2 reasons
pseudofinder_only_filter = pseudofinder_only[pseudofinder_only['reasons_count'] >= 2]
pseudofinder_only_filter.head(2)

In [156]:
Sodalis_emapper_transporters = Sodalis_emapper[(Sodalis_emapper['BRITE'].str.contains('ko02000', na=False))][['query','COG_category','Preferred_name','KEGG_ko','ko_symbol',
                                                                                                            'ko_name','module','PFAMs','Description']].drop_duplicates().sort_values(by=['query','COG_category']).reset_index(drop=True)
#Sodalis_transporters.to_csv('Sodalis_transporters_annotations.tsv', sep='\t', index=False)
Sodalis_emapper_LPS = Sodalis_emapper[(Sodalis_emapper['BRITE'].str.contains('ko01005', na=False)) |
                                       (Sodalis_emapper['KEGG_Pathway'].str.contains('ko00540', na=False))][['query','COG_category',
                                                                                                             'Description','Preferred_name','EC','KEGG_ko','ko_symbol','ko_name','BRITE','PFAMs','taxa_scope']].drop_duplicates().sort_values(by=['query','COG_category']).reset_index(drop=True)
Sodalis_emapper_PTG = Sodalis_emapper[(Sodalis_emapper['BRITE'].str.contains('ko01011', na=False)) |
                                      (Sodalis_emapper['KEGG_Pathway'].str.contains('ko00550', na=False))][['query','COG_category','Description','Preferred_name','EC','KEGG_ko',
                                                                                                            'ko_symbol','ko_name','BRITE','PFAMs','taxa_scope']].drop_duplicates().sort_values(by=['query','COG_category']).reset_index(drop=True)
Sodalis_emapper_AminoS = Sodalis_emapper[(Sodalis_emapper['KEGG_Pathway'].str.contains('ko00520', na=False))][['query','COG_category','Description','Preferred_name','EC','KEGG_ko',
                                                                                                               'ko_symbol','ko_name','BRITE','PFAMs','taxa_scope']].drop_duplicates().sort_values(by=['query','COG_category']).reset_index(drop=True)

<a id='data'></a>
## Data analysis: General Profile
* [Sessions list](#sessions)

In [ ]:
modules['module'].nunique() 
#modules[(modules['enzymes_uniq'] != 'No enzymes unique to module')]['module'].nunique() 
#modules_complete['module'].nunique() 
#modules_incomplete['module'].nunique() 
#modules_low['module'].nunique() 

119

In [95]:
hits.groupby('contig')['enzyme'].count().reset_index(name='ko_count').sort_values(by='ko_count', ascending=False).head()

,contig,ko_count
10,c_000000000011,104
2,c_000000000003,82
12,c_000000000013,70
4,c_000000000005,70
7,c_000000000008,57


In [ ]:
#EC_annotations = all_annotations[(all_annotations['source'] == 'EGGNOG_EC_NUMBER') & (all_annotations['e_value'] < 1e-5)]['function'].unique() 
# # EGGNOG_EC_NUMBER 307, EGGNOG_BRITE 134, EGGNOG_BiGG_REACTIONS 279, EGGNOG_KEGG_REACTION 107
#all_annotations['source'].unique()
#all_annotations[(all_annotations['source'] == 'EGGNOG_BEST_TAX') & (all_annotations['function'].str.contains('Entero'))]
#all_annotations[(all_annotations['source'] == 'EGGNOG_COG_CATEGORY') & (all_annotations['e_value'] < 1e-50)].groupby('function').size().reset_index(name='counts').sort_values(by='counts', ascending=False).head()

In [ ]:
Transporters_kos_list = Sodalis_emapper_transporters['KEGG_ko'].unique().tolist()
folate_kos_list = Sodalis_emapper[Sodalis_emapper['KEGG_Pathway'].str.contains('ko00790|ko00670')]['KEGG_ko'].unique().tolist()
LPS_kos_list = Sodalis_emapper_LPS['KEGG_ko'].unique().tolist() 
PTG_kos_list = Sodalis_emapper_PTG['KEGG_ko'].unique().tolist() 
AminoS_kos_list = Sodalis_emapper_AminoS['KEGG_ko'].unique().tolist() 

In [ ]:
Sodalis_emapper[Sodalis_emapper['KEGG_ko'].isin(LPS_kos_list + PTG_kos_list)].head(2)

In [ ]:
#insecta_annotations['module'].nunique() 
insecta_annotations['KEGG_ko'].nunique() 

4529

In [241]:
manual_pseudogenes.head(2)

,query,Disruption,Coverage,Cov_comp,Variant_Type,DP,Preferred_name,COG_category,EC,KEGG_ko,module,Description,ko_symbol,ko_name,module_name,module_class,module_pathway,KEGG_Pathway,BRITE
0,MRSS_00006,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00019,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00290 Valine, leucine and isoleucine biosynthesis","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"
1,MRSS_00006,Truncated; Frameshift/Internal Stop,17.69,Low Cov,-,-,ilvE,E,2.6.1.42,K00826,M00036,Belongs to the class-IV pyridoxal-phosphate-dependent aminotransferase family,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],"Leucine degradation, leucine => acetoacetate + acetyl-CoA",Pathway modules; Amino acid metabolism; Branched-chain amino acid metabolism,"map00280 Valine, leucine and isoleucine degradation","ko00270,ko00280,ko00290,ko00770,ko01100,ko01110,ko01130,ko01210,ko01230,map00270,map00280,map00290,map00770,map01100,map01110,map01130,map01210,map01230","ko00000,ko00001,ko00002,ko01000,ko01007"


In [239]:
manual_pseudogenes_kos_list = manual_pseudogenes['KEGG_ko'].unique().tolist()
manual_pseudogenes_module_list = manual_pseudogenes['module'].unique().tolist()
manual_pseudogenes_query_list = manual_pseudogenes['query'].unique().tolist()
manual_pseudogenes[['query','Coverage','Cov_comp','Variant_Type','Preferred_name','KEGG_ko','ko_symbol','ko_name','module','module_name']].drop_duplicates().head(2)
manual_pseudogenes['query'].nunique()

80

In [240]:
comparison_manual_pseudo_filter[['query','Coverage','Cov_comp','Variant_Type','Preferred_name','KEGG_ko','ko_symbol','ko_name','module','module_name']].drop_duplicates().head(2)

,query,Coverage,Cov_comp,Variant_Type,Preferred_name,KEGG_ko,ko_symbol,ko_name,module,module_name
0,MRSS_00006,17.69,Low Cov,-,ilvE,K00826,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],M00019,"Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine"
1,MRSS_00006,17.69,Low Cov,-,ilvE,K00826,"E2.6.1.42, ilvE",branched-chain amino acid aminotransferase [EC:2.6.1.42],M00036,"Leucine degradation, leucine => acetoacetate + acetyl-CoA"


In [ ]:
comparison_manual_pseudofinder['Identification_Source'].unique() # 'Match', 'Manual Only', 'Pseudofinder Only
comparison_manual_pseudo_filter['query'].nunique() 
comparison_manual_pseudo_filter['KEGG_ko'].nunique() 
#comparison_manual_pseudo_filter['module'].nunique() 
#pseudofinder_only['module'].nunique() 

68

In [206]:
comparison_filter_kos_list = comparison_manual_pseudo_filter['KEGG_ko'].unique().tolist()
comparison_filter_module_list = comparison_manual_pseudo_filter['module'].unique().tolist()
comparison_filter_query_list = comparison_manual_pseudo_filter['query'].unique().tolist()

In [243]:
comparison_manual_pseudo_filter[['query','COG_category']].drop_duplicates().groupby('COG_category')['query'].count().reset_index(name='COG_count').sort_values(by='COG_count', ascending=False).head()

,COG_category,COG_count
14,M,13
8,H,12
3,E,9
16,S,7
10,J,5


In [ ]:
pseudofinder_only_filter['query'].nunique() # 17
pseudofinder_only_filter['module'].nunique() # 13
pseudofinder_only_filter['KEGG_ko'].nunique() # 18

18

In [244]:
pseudofinder_only_filter[['query','COG_category']].drop_duplicates().groupby('COG_category')['query'].count().reset_index(name='COG_count').sort_values(by='COG_count', ascending=False).head()

,COG_category,COG_count
7,J,6
3,F,2
0,C,1
2,E,1
1,DM,1


In [245]:
pseudofinder_only_filter.groupby('module')['KEGG_ko'].count().reset_index(name='kos_count').sort_values(by='kos_count', ascending=False).head()

,module,kos_count
0,-,27
9,M00360,10
8,M00359,10
6,M00178,7
5,M00050,6


### Modules X Pseudogenes map

In [211]:
# modules_complete_list, modules_incomplete_list, modules_low_list
comparison_manual_pseudo_filter[comparison_manual_pseudo_filter['module'].isin(modules_complete_list)]
comparison_manual_pseudo_filter[comparison_manual_pseudo_filter['module'].isin(modules_incomplete_list)][['query','EC','KEGG_ko','ko_symbol',
                                                                                                  'ko_name','module','module_name','module_class','module_pathway']].head(2)
#comparison_manual_pseudofinder[comparison_manual_pseudofinder['module'].isin(modules_low_list)][['query','EC','KEGG_ko','ko_symbol','ko_name','module','module_name','module_class','module_pathway','taxa_scope','Identification_Source']].head()

,query,EC,KEGG_ko,ko_symbol,ko_name,module,module_name,module_class,module_pathway
22,MRSS_00023,2.5.1.54,K01626,"E2.5.1.54, aroF, aroG, aroH",3-deoxy-7-phosphoheptulonate synthase [EC:2.5.1.54],M00022,"Shikimate pathway, phosphoenolpyruvate + erythrose-4P => chorismate",Pathway modules; Amino acid metabolism; Aromatic amino acid metabolism,"map00400 Phenylalanine, tyrosine and tryptophan biosynthesis"
23,MRSS_00023,2.5.1.54,K01626,"E2.5.1.54, aroF, aroG, aroH",3-deoxy-7-phosphoheptulonate synthase [EC:2.5.1.54],M00022,"Shikimate pathway, phosphoenolpyruvate + erythrose-4P => chorismate",Pathway modules; Amino acid metabolism; Aromatic amino acid metabolism,"map00400 Phenylalanine, tyrosine and tryptophan biosynthesis"


In [ ]:
# Check extra mmodules
#Sodalis_emapper[
#	(~Sodalis_emapper['module'].isin(modules_complete_final_list)) & 
#	(~Sodalis_emapper['module'].isin(modules_incomplete_final_list)) &
 #   (~Sodalis_emapper['module'].isin(modules_low_final_list))
#].groupby('module')['KEGG_ko'].count().reset_index(name='ko_hits').sort_values(by='ko_hits', ascending=False).head(2)

,module,ko_hits
0,-,389
61,No_entry_found,247


In [179]:
modules_complete['module'].unique() # M00118,M00307,M00007,M00417,M00157,M00579,M00060,M00083,M00125,M00120,M00881,M00049,M00053,M00938
modules_incomplete['module'].unique() # M00001,M00003,M00909,M00866,M00064,M00123,M00126,M00051
#modules_incomplete[modules_incomplete['module'] == 'M00003']
#modules_low[modules_low['module'] == 'M00525']
modules_complete_final_list = ['M00118','M00307','M00007','M00417','M00157','M00579','M00060','M00083','M00125','M00120','M00881','M00049','M00053','M00938','M00995','M00999','M00909']
modules_incomplete_final_list = ['M00001','M00002','M00003','M00866','M00064','M00123','M00126','M00051','M00761','M00082','M00093','M00052','M00005','M00140','M00549']
modules_low_final_list = ['M00023','M00022','M00117','M00016','M00050','M00096','M00527','M00526','M00988','M00004','M00572','M00063','M00364','M00700'] # M00700: 'K18104'
modules[modules['module'] == 'M00064']

,module,module_name,module_category,module_subcategory,module_definition,step_compl,path_compl,prop_enzymes_uniq,enzymes_uniq,enzyme_hits_in_module,warnings
65,M00064,ADP-L-glycero-D-manno-heptose biosynthesis,Glycan metabolism,Lipopolysaccharide metabolism,"K03271 (K03272,K21344) K03273 (K03272,K21345) K03274",0.8,0.8,1.0,"K03271,K03272,K03274","K03271,K03272,K03274",NaN


In [215]:
manual_pseudogenes[manual_pseudogenes['module'].isin(modules_complete_final_list)]
comparison_manual_pseudo_filter[comparison_manual_pseudo_filter['module'].isin(modules_complete_final_list)]
pseudofinder_only_filter[pseudofinder_only_filter['module'].isin(modules_complete_final_list)]

,query,reasons_count,Pseudofinder_reasons,Coverage,Cov_comp,Variant_Type,DP,Preferred_name,COG_category,EC,KEGG_ko,module,Description,ko_symbol,ko_name,module_name,module_class,module_pathway,KEGG_Pathway,BRITE
46,MRSS_00014,2,Intergenic region with 15 blast hits.;Predicted fragmentation of a single gene.,27.07,Mean Cov,-,-,pykF,G,2.7.1.40,K00873,M00049,Belongs to the pyruvate kinase family,"PK, pyk",pyruvate kinase [EC:2.7.1.40],"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP",Pathway modules; Nucleotide metabolism; Purine metabolism,map00230 Purine metabolism,"ko00010,ko00230,ko00620,ko01100,ko01110,ko01120,ko01130,ko01200,ko01230,ko04922,ko04930,ko05165,ko05203,ko05230,map00010,map00230,map00620,map01100,map01110,map01120,map01130,map01200,map01230,map04922,map04930,map05165,map05203,map05230","ko00000,ko00001,ko00002,ko01000,ko04131,ko04147"
50,MRSS_00014,2,Intergenic region with 15 blast hits.;Predicted fragmentation of a single gene.,27.07,Mean Cov,-,-,pykF,G,2.7.1.40,K00873,M00049,Belongs to the pyruvate kinase family,"PK, pyk",pyruvate kinase [EC:2.7.1.40],"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP",Pathway modules; Nucleotide metabolism; Purine metabolism,map00230 Purine metabolism,"ko00010,ko00230,ko00620,ko01100,ko01110,ko01120,ko01130,ko01200,ko01230,ko04922,ko04930,ko05165,ko05203,ko05230,map00010,map00230,map00620,map01100,map01110,map01120,map01130,map01200,map01230,map04922,map04930,map05165,map05203,map05230","ko00000,ko00001,ko00002,ko01000,ko04131,ko04147"


### Check specific routes or enzyme sets

In [249]:
#extra_modules_kos = ['K00525','K00526','K00560','K00940','K00943','K01520','K00075','K00790','K00820','K03431','K04042','K00012','K08679','K01495','K01737','K06879','K06920','K09457','K10026','K18104'] # M00988, M00999, M00995
#ko_list = ['K02062', 'K02063', 'K02064','K02065','K02066','K02067','K06861','K07091','K07122','K07323','K09690','K09691','K09808','K09810','K09815','K09816','K09817','K09996','K09997','K09998','K09999','K10000','K10014','K11085','K11720','K18104','K18889','K18890'] # 02010 ABC transporters
#ko_list = ['K02746','K02777','K02784','K02793','K02794','K02795','K02796','K02806','K08483'] # 02060 Phosphotransferase system (PTS)
#ko_list = ['K03070','K03071','K03072','K03073','K03074','K03075','K03076','K03106','K03110','K03210','K03217'] # 03070 Bacterial secretion system
#ko_list = ['K03179', 'K03182', 'K03184', 'K03185']
#ko_list = ['K01624','K00850'] # Formaldehyde assimilation
#ko_list = ['K00287','K00600','K00560'] # Folate transport and metabolism
ko_list = ['K18590', 'K07660', 'K18104', 'K09476'] # BRITE AMR, only K07660
#ko_list =['K18590', 'K07660', 'K18104', 'K18890', 'K09475', 'K09476', 'K14062', 'K00287'] # BRITE AMR, only K07660 in comparison_manual_pseudofinder
#ko_list = ['K06223','K03531','K03569','K02621','K02622','K01885','K04487'] # Prokaryotic defense system BRITE ko02048
#pseudofinder_kos = ['K00826','K00606','K01918','K00077','K01626','K01735','K01736','K01657','K01665','K13950','K02619','K03797','K15257','K07660','K03184','K18800','K03688','K05851','K03591','K02777',
#                    'K00948','K01963','K01613','K01937','K00873','K01834','K00195','K00088','K13014','K01835','K00014','K01714','K00928','K00991','K00919','K01770','K06041','K01627','K00833','K03599',
#                    'K03179','K03186','K03183','K11754','K06879','K09457','K01737','K03286','K07287','K07278','K09800','K22051','K05802','K03313','K03324','K06189','K04758','K02784','K08483','K18691']
comparison_manual_pseudo_filter[comparison_manual_pseudo_filter['KEGG_ko'].isin(ko_list)]['KEGG_ko'].unique()
# [['query','Note','COG_category','KEGG_ko','ko_symbol','ko_name','module','Description']].drop_duplicates()


array(['K07660'], dtype=object)

In [248]:
pseudofinder_only_filter[pseudofinder_only_filter['KEGG_ko'].isin(ko_list)]['KEGG_ko'].unique()
#pseudofinder_only_filter[pseudofinder_only_filter['KEGG_ko'].isin(ko_list)][['query','reasons_count','COG_category','KEGG_ko','ko_symbol','ko_name','module','Note']].drop_duplicates().sort_values(by=['reasons_count']).reset_index(drop=True)

array([], dtype=object)

In [247]:
Sodalis_emapper[Sodalis_emapper['KEGG_ko'].isin(ko_list)][['query','COG_category','KEGG_ko','ko_symbol','ko_name','Description']].drop_duplicates()

,query,COG_category,KEGG_ko,ko_symbol,ko_name,Description
247,MRSS_00217,L,K02622,parE,topoisomerase IV subunit B [EC:5.6.2.2],Topoisomerase IV is essential for chromosome segregation. It relaxes supercoiled DNA. Performs the decatenation events required during the replication of a circular DNA molecule
489,MRSS_00472,J,K01885,"EARS, gltX",glutamyl-tRNA synthetase [EC:6.1.1.17],Catalyzes the attachment of glutamate to tRNA(Glu) in a two-step reaction glutamate is first activated by ATP to form Glu-AMP and then transferred to the acceptor end of tRNA(Glu)
578,MRSS_00538,D,K03531,ftsZ,cell division protein FtsZ,Essential cell division protein that forms a contractile ring structure (Z ring) at the future cell division site. The regulation of the ring assembly controls the timing and the location of cell division. One of the functions of the FtsZ ring is to recruit other cell division proteins to the septum to produce a new cell wall between the dividing cells. Binds GTP and shows GTPase activity
621,MRSS_00569,D,K03569,mreB,rod shape-determining protein MreB and related proteins,Rod shape-determining protein
734,MRSS_00676,L,K02621,parC,topoisomerase IV subunit A [EC:5.6.2.2],Topoisomerase IV is essential for chromosome segregation. It relaxes supercoiled DNA. Performs the decatenation events required during the replication of a circular DNA molecule
843,MRSS_00781,L,K06223,dam,DNA adenine methylase [EC:2.1.1.72],Site-specific DNA-methyltransferase (Adenine-specific)
921,MRSS_00856,E,K04487,"iscS, NFS1",cysteine desulfurase [EC:2.8.1.7],"Master enzyme that delivers sulfur to a number of partners involved in Fe-S cluster assembly, tRNA modification or cofactor biosynthesis. Catalyzes the removal of elemental sulfur atoms from cysteine to produce alanine. Functions as a sulfur delivery protein for Fe-S cluster synthesis onto IscU, an Fe-S scaffold assembly protein, as well as other S acceptor proteins"


### Transporters and other BRITE and KEGG Pathways

In [250]:
# Get KEGG_ko values from Sodalis_transporters not present in pseudofinder_transporters
Sodalis_transporters_intact = Sodalis_emapper_transporters[~(Sodalis_emapper_transporters['KEGG_ko'].isin(manual_pseudogenes_kos_list)) &
                                                           ~(Sodalis_emapper_transporters['KEGG_ko'].isin(comparison_manual_pseudo_kos_list)) &
                                                           ~(Sodalis_emapper_transporters['KEGG_ko'].isin(pseudofinder_only_filter['KEGG_ko']))].sort_values(by=['KEGG_ko']).reset_index(drop=True)
Sodalis_transporters_intact_ko_list = Sodalis_transporters_intact['KEGG_ko'].unique().tolist()
Sodalis_transporters_intact.head(2)

,query,COG_category,Preferred_name,KEGG_ko,ko_symbol,ko_name,module,PFAMs,Description
0,MRSS_00656,P,thiQ,K02062,thiQ,thiamine transport system ATP-binding protein [EC:7.6.2.15],No_entry_found,ABC_tran,Part of the ABC transporter complex ThiBPQ involved in thiamine import. Responsible for energy coupling to the transport system
1,MRSS_00655,P,thiP,K02063,thiP,thiamine transport system permease protein,No_entry_found,BPD_transp_1,with TbpA and ThiQ functions in transport of thiamine and thiamine pyrophosphate into the cell


In [ ]:
Sodalis_emapper_transporters[Sodalis_emapper_transporters['KEGG_ko'].isin(['K09808','K09810'])] # 'K07091','K11720','K06861' LptFGB lipopolysaccharide export system
# 'K09690','K09691' RfbAB O-antigen export system COG GM
# 'K09808','K09810' LolCEFD Lipoprotein releasing system transmembrane

,query,COG_category,Preferred_name,KEGG_ko,ko_symbol,ko_name,module,PFAMs,Description
87,MRSS_00825,M,lolE,K09808,lolC_E_F,lipoprotein-releasing system permease protein,No_entry_found,"FtsX,MacB_PCD",Lipoprotein releasing system transmembrane protein
88,MRSS_00826,V,lolD,K09810,lolD,lipoprotein-releasing system ATP-binding protein [EC:7.6.2.-],No_entry_found,ABC_tran,Part of the ABC transporter complex LolCDE involved in the translocation of
89,MRSS_00827,M,lolC,K09808,lolC_E_F,lipoprotein-releasing system permease protein,No_entry_found,"FtsX,MacB_PCD",Lipoprotein releasing system transmembrane protein


In [233]:
Sodalis_emapper_transporters[
	(Sodalis_emapper_transporters['KEGG_ko'].isin(manual_pseudogenes_kos_list)) |
	(Sodalis_emapper_transporters['KEGG_ko'].isin(comparison_manual_pseudo_kos_list)) |
    (Sodalis_emapper_transporters['KEGG_ko'].isin(pseudofinder_only_filter['KEGG_ko']))][['query','COG_category','KEGG_ko',
                                                                                      'ko_symbol','ko_name','Description']].drop_duplicates().sort_values(by=['Description','ko_symbol'])

,query,COG_category,KEGG_ko,ko_symbol,ko_name,Description
56,MRSS_00459,GM,K09691,"wzt, rfbB",homopolymeric O-antigen transport system ATP-binding protein [EC:7.5.2.14],ABC transporter
50,MRSS_00405,M,K03286,TC.OOP,"OmpA-OmpF porin, OOP family",Belongs to the ompA family
61,MRSS_00568,M,K03286,TC.OOP,"OmpA-OmpF porin, OOP family",Belongs to the ompA family
92,MRSS_00871,M,K03286,TC.OOP,"OmpA-OmpF porin, OOP family",Belongs to the ompA family
29,MRSS_00236,M,K03282,mscL,large conductance mechanosensitive channel,Channel that opens in response to stretch forces in the membrane lipid bilayer. May participate in the regulation of osmotic pressure changes within the cell
44,MRSS_00354,P,K06189,"corC, tlyC",hemolysin (HlyC) family protein,Mg2 and Co2 transporter CorC
1,MRSS_00018,P,K03313,nhaA,"Na+:H+ antiporter, NhaA family",Na( ) H( ) antiporter that extrudes sodium in exchange for external protons
0,MRSS_00009,M,K07287,bamC,outer membrane protein assembly factor BamC,"Part of the outer membrane protein assembly complex, which is involved in assembly and insertion of beta-barrel proteins into the outer membrane"
9,MRSS_00080,M,K05802,"mscK, kefA, aefA",potassium-dependent mechanosensitive channel,mechanosensitive ion channel
10,MRSS_00080,M,K22051,"mscM, bspA",miniconductance mechanosensitive channel,mechanosensitive ion channel


In [251]:
pseudofinder_only_filter[pseudofinder_only_filter['KEGG_Pathway'].str.contains('00670', na=False)][
    ['query','COG_category','KEGG_ko','ko_symbol','ko_name','Description']
].drop_duplicates().sort_values(by=['COG_category','query']).reset_index(drop=True)['KEGG_ko'].unique()

array([], dtype=object)

In [252]:
#pseudo_comp_Sodalis_transporters[['query','COG_category','KEGG_ko','ko_symbol','ko_name','BRITE_Level_4','BRITE_Level_6','BRITE_Level_7','BRITE_Level_8']].drop_duplicates()
comparison_manual_pseudo_filter[comparison_manual_pseudo_filter['KEGG_Pathway'].str.contains('00670', na=False)][ # ko02044, 
    ['query','COG_category','KEGG_ko','ko_symbol','ko_name','Description']
].drop_duplicates().sort_values(by=['COG_category','query']).reset_index(drop=True)

,query,COG_category,KEGG_ko,ko_symbol,ko_name,Description


### Sodalis X Insecta

In [262]:
insecta_kos_definitions = pd.read_csv('insecta_annotations_kos_definitions.txt', sep=",", names=['KEGG_ko','ko_symbol','ko_name'])
insecta_kos_definitions.head()

,KEGG_ko,ko_symbol,ko_name
0,KEGG_ko,ko_symbol,ko_name
1,K13356,FAR,alcohol-forming fatty acyl-CoA reductase [EC:1.2.1.84]
2,K09355,PBX1,pre-B-cell leukemia transcription factor 1
3,K14327,"UPF2, RENT2",regulator of nonsense transcripts 2
4,K07197,"SREBP1, SREBF1",sterol regulatory element-binding transcription factor 1


In [264]:
insecta_annotations[(insecta_annotations['BRITE'].str.contains('ko02000')) & (insecta_annotations['KEGG_ko'].isin(insecta_kos_definitions['KEGG_ko']))]
insecta_annotations.head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
1,k141_100027_0,I,-,1.2.1.84,K13356,-,-,"NAD_binding_4,Sterile",Catalyzes the reduction of fatty acyl-CoA to fatty alcohols,Hymenoptera,"ko00073,ko04146,ko04212,map00073,map04146,map04212","ko00000,ko00001,ko01000"
2,k141_100050_0,K,exd,-,K09355,-,-,"Homeobox,PBC",PBC domain,Paraneoptera,"ko04927,ko04934,ko05202,map04927,map04934,map05202","ko00000,ko00001,ko03000"


In [265]:
insecta_annotations[insecta_annotations['BRITE'].str.contains('ko02000')].head()
insecta_annotations[insecta_annotations['BRITE'].str.contains('ko02000')][['query','COG_category','Description','Preferred_name','EC','KEGG_ko']].drop_duplicates().head(2)

,query,COG_category,Description,Preferred_name,EC,KEGG_ko
33,k141_100410_0,E,Solute carrier family 12,-,-,K14429
41,k141_100652_0,P,Piezo,-,-,K22128


In [268]:
insecta_annotations[insecta_annotations['KEGG_ko'] == 'K08186']
insecta_annotations[insecta_annotations['module'] == 'M00121'].head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
18,k141_100260_0,H,-,1.3.3.3,K00228,M00121,-,Coprogen_oxidas,Coproporphyrinogen III oxidase,Hymenoptera,"ko00860,ko01100,ko01110,map00860,map01100,map01110","ko00000,ko00001,ko00002,ko01000"
352,k141_105612_0,J,-,"6.1.1.15,6.1.1.17",K14163,M00121,-,"GST_C,GST_C_3,HGTP_anticodon,ProRS-C_1,WHEP-TRS,tRNA-synt_1c,tRNA-synt_1c_C,tRNA-synt_2b","tRNA synthetases class I (E and Q), catalytic domain",Hymenoptera,"ko00860,ko00970,ko01100,ko01110,ko01120,map00860,map00970,map01100,map01110,map01120","ko00000,ko00001,ko00002,ko01000,ko03016"


In [ ]:
insecta_annotations[insecta_annotations['KEGG_ko'].isin(comparison_filter_kos_list)].head(2)

,query,COG_category,Preferred_name,EC,KEGG_ko,module,CAZy,PFAMs,Description,taxa_scope,KEGG_Pathway,BRITE
746,k141_112373_0,F,-,-,K07071,-,-,"DUF1731,Epimerase",Domain of unknown function (DUF1731),Nematocera,-,ko00000
3545,k141_15759_0,H,-,"4.99.1.1,4.99.1.9",K01772,M00121,-,Ferrochelatase,Catalyzes the ferrous insertion into protoporphyrin IX,Hymenoptera,"ko00860,ko01100,ko01110,map00860,map01100,map01110","ko00000,ko00001,ko00002,ko01000"


In [254]:
insecta_annotations[insecta_annotations['module'].isin(modules_incomplete_final_list)][['query','COG_category','Description','Preferred_name','EC','KEGG_ko','module']].drop_duplicates().sort_values(by=['module','query']).reset_index(drop=True).head(2)

,query,COG_category,Description,Preferred_name,EC,KEGG_ko,module
0,k141_104190_0,G,Belongs to the pyruvate kinase family,-,2.7.1.40,K00873,M00001
1,k141_127603_0,G,Belongs to the hexokinase family,Hex-A,2.7.1.1,K00844,M00001
